In [1]:
from pathlib import Path
import sys
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))

from src.data_download import download_data
import json 
import pandas as pd
import pickle
import os

data_dir = Path("../data")

In [2]:
# List the categories to download here
categories = ["All_Beauty"]
reload_data = False

if reload_data:
    for category in categories:
        download_data(category)


In [3]:
for category in categories:
    review_file = f"../data/raw/{category}.jsonl"
    metadata_file = f"../data/raw/meta_{category}.jsonl"

    print(f"Data exploration for {category} dataset:")

    # Read reviews
    reviews = []
    with open(review_file, "r") as f:
        for line in f:
            reviews.append(json.loads(line))

    # Read metadata
    meta = []
    with open(metadata_file, "r") as f:
        for line in f:
            meta.append(json.loads(line))

    print("Reviews count:", len(reviews))
    print("Metadata count:", len(meta))

    print("Review fields:", list(reviews[0].keys()))
    print("Metadata fields:", list(meta[0].keys()))

    print("First review:", reviews[0])
    print("First metadata record:", meta[0])


Data exploration for All_Beauty dataset:
Reviews count: 701528
Metadata count: 112590
Review fields: ['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']
Metadata fields: ['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together']
First review: {'rating': 5.0, 'title': 'Such a lovely scent but not overpowering.', 'text': "This spray is really nice. It smells really good, goes on really fine, and does the trick. I will say it feels like you need a lot of it though to get the texture I want. I have a lot of hair, medium thickness. I am comparing to other brands with yucky chemicals so I'm gonna stick with this. Try it!", 'images': [], 'asin': 'B00YQ6X8EO', 'parent_asin': 'B00YQ6X8EO', 'user_id': 'AGKHLEW2SOWHNMFQIJGBECAF7INQ', 'timestamp': 1588687728923, 'helpful_vote': 0, 'verified_purchase': Tr

In [4]:
from langchain_core.documents import Document
from src.bm25 import text_tokenizer, build_bm25, bm25_search

tokenized_corpus_file = os.path.join(data_dir, "processed", "tokenized_corpus.pkl")

documents = []
tokenized_corpus = []

metadata_lookup = {
    item["parent_asin"]: item
    for item in meta
}

if os.path.exists(tokenized_corpus_file):
    print("Loading tokenized corpus...")
    
    with open(tokenized_corpus_file, "rb") as f:
        tokenized_corpus = pickle.load(f)

else:
    print("Generating tokenized corpus...")
    
    for review in reviews:
        tokens = text_tokenizer(review["text"])
        tokenized_corpus.append(tokens)

    with open(tokenized_corpus_file, "wb") as f:
        pickle.dump(tokenized_corpus, f)

# Build documents (needed for LangChain BM25)
for tokens, review in zip(tokenized_corpus, reviews):
    processed_text = " ".join(tokens)
    product_metadata = metadata_lookup.get(review["parent_asin"], "")
    combined_text = product_metadata["title"] + " " + processed_text
    documents.append(
        Document(
            page_content=combined_text,
            metadata={"asin": review["parent_asin"],
            "product_review": review["text"],
            "product_title": product_metadata["title"],
            "product_rating": review["rating"]
            }
        )
    )


Loading tokenized corpus...


In [5]:
bm25_index_path = "../data/processed/bm25_index.pkl"
rebuild_bm25 = True

if os.path.exists(bm25_index_path) and not rebuild_bm25:

    print("Loading existing BM25 index...")

    with open(bm25_index_path, "rb") as f:
        bm25 = pickle.load(f)

else:

    print("Building BM25 index...")

    bm25 = build_bm25(documents)

    with open(bm25_index_path, "wb") as f:
        pickle.dump(bm25, f)

Building BM25 index...


In [6]:
# This is how you pass a query to the BM25 search

query = "wireless bluetooth headphones"

results = bm25_search(bm25, documents, query, k=5)

for doc, score in results:
    print("Product Title:", doc.metadata.get("product_title"))
    print("Product Review:", doc.metadata.get("product_review")[:200])
    print("ASIN:", doc.metadata.get("asin"))
    print("Product Rating:", doc.metadata.get("product_rating"))
    print("Retrieval Score:", score)
    print()

Product Title: Wireless Earbuds, 455D Stereo Sound Wireless Headphones Wireless Sport Earbud with Breathing Mini in-Ear Sports Earphones Noise Cancelling Headsets, Bluetooth Earbuds
Product Review: Works fine...
ASIN: B014VTGC9I
Product Rating: 5.0
Retrieval Score: 35.066935752274375

Product Title: Yontune Sleep Headphones Headband Timing Wireless Cozy Band Washable for Running Workout Unique Gifts (Lengthened), Black grey
Product Review: Bluetooth didn’t turn on
ASIN: B0B533WQVC
Product Rating: 1.0
Retrieval Score: 31.791503280805657

Product Title: FENCHILIN Vanity Mirror with Lights Bluetooth Lighted Makeup Mirror Touch Screen Wireless Audio Speaker Dimmable Light Detachable 10X Magnification Rechargable Power (Rose Gold)
Product Review: Love the mirror and light's.  Has bluetooth. Can change mirror different positions.
ASIN: B0769VLLW6
Product Rating: 5.0
Retrieval Score: 21.152462808027963

Product Title: FENCHILIN Vanity Mirror with Lights Bluetooth Lighted Makeup Mirror Touch S

In [7]:
# This is how you pass a query to the semantic search

from src.semantic import semantic_search

query = "wireless bluetooth headphones"

results = semantic_search(documents, query, k=5, sample_size=10000, reload_index=True)

for doc, score in results:
    print("Product Title:", doc.metadata.get("product_title"))
    print("Product Review:", doc.metadata.get("product_review")[:200])
    print("ASIN:", doc.metadata.get("asin"))
    print("Product Rating:", doc.metadata.get("product_rating"))
    print("Retrieval Score:", score)
    print()

Product Title: Earbuds Ear Buds Sport Earbuds Running Earbuds in Ear Headphones Wired Earphones with Microphone Mic Stereo and Volume Control Waterproof Wired Earphone Android Mp3 Players Tablet Laptop 3.5mm Audio
Product Review: Other reviews are not for this product. Does not fit well in ear and doesn't come with any way to change fit/size. Can hardly wear when sitting on the couch so there's no way they'd work for running. 
ASIN: B07KQ4XNDV
Product Rating: 1.0
Retrieval Score: 1.0159402

Product Title: MMUSS Sleep Headphones Headband with Ultra Thin Stereo Speakers.Perfect for Sleeping,Sports,Air Travel,Meditation and Relaxation(Black2)
Product Review: I love these. Very comfortable and great sound quality.
ASIN: B07JQ136H9
Product Rating: 5.0
Retrieval Score: 1.1151774

Product Title: Gaming Headphones Chamvict Xbox One Headset with Stereo Sound Ps4 Headset with Mic Noise Canceling for PS4,PC,Laptop,Cell Phone,Xbox,Find Enemies Before They Find You
Product Review: The quality of th

In [8]:
queries = [
    # Easy (Keyword-based)
    "ultra facial barrier-hydrating cleanser",
    "AM Facial Moisturizing Lotion SPF 30",
    "fit me concealer",
    "cheek heat gel cream blush, face makeup",
    "oil control moisturizing gel-cream"
    # Medium (Semantic-based)
    "something to keep your face moisturized all day",
    "makeup to cover up pimples",
    "comfortable lotion for rigid weather",
    # Complex
    "best sunscreen for scuba diving in tropical regions",
    "what’s the best hair treatment to prevent hair loss",
    "good cleanser for busy working professionals who do not have time"
]

all_results = {}

for query in queries:
    bm25_results = bm25_search(bm25, documents, query, k=5)
    semantic_results = semantic_search(
        documents,
        query,
        k=5,
        sample_size=10000,
        reload_index=False   # usually better after first build
    )
    
    all_results[query] = {
        "bm25": bm25_results,
        "semantic": semantic_results
    }

In [9]:
test_query = queries[0]
print(all_results[test_query]["bm25"][0])
print(all_results[test_query]["semantic"][0])

(Document(metadata={'asin': 'B00U2VQZC4', 'product_review': 'Good facial cleanser for my sensitive skin', 'product_title': 'Neutrogena Ultra Light Facial Cleansing Oil & Makeup Remover, Non-Comedogenic Face Oil Cleanser to Remove Dirt, Oil, Makeup & Waterproof Mascara, 4 fl. oz', 'product_rating': 5.0}, page_content='Neutrogena Ultra Light Facial Cleansing Oil & Makeup Remover, Non-Comedogenic Face Oil Cleanser to Remove Dirt, Oil, Makeup & Waterproof Mascara, 4 fl. oz good facial cleanser my sensitive skin'), 17.792769631882074)
(Document(id='aa87ed2d-a2c1-435c-82d0-92149fa64ba4', metadata={'asin': 'B09ZDQ626L', 'product_review': 'I like the silicone brush, it’s soft and scrubs without scratching.  I wish it comes off, was kinda weird holding the whole bottle to scrub my face.<br />The cleanser looks and smells like unscented liquid soap.  It is like cleaning my face with foaming hand soap. It left it squeaky clean, which is nice. Takes away all the excess oil and grime and makeup.  M

In [26]:
def deduplicate_results(results, key="asin"):
    seen = set()
    unique_results = []

    for r in results:
        value = r.get(key, None)
        if value not in seen:
            seen.add(value)
            unique_results.append(r)

    return unique_results

In [23]:
def format_result(result):
    doc, score = result
    
    metadata = getattr(doc, "metadata", {})
    page_content = getattr(doc, "page_content", "")

    return {
        "title": metadata.get("product_title", "N/A"),
        "asin": metadata.get("asin", "N/A"),
        "score": float(score)
    }

In [27]:
formatted_results = {}

for query, result_dict in all_results.items():
    bm25_formatted = [format_result(r) for r in result_dict["bm25"]]
    semantic_formatted = [format_result(r) for r in result_dict["semantic"]]

    formatted_results[query] = {
        "bm25": deduplicate_results(bm25_formatted, key="asin"),
        "semantic": deduplicate_results(semantic_formatted, key="asin"),
    }

In [28]:
formatted_results[queries[0]]

{'bm25': [{'title': 'Neutrogena Ultra Light Facial Cleansing Oil & Makeup Remover, Non-Comedogenic Face Oil Cleanser to Remove Dirt, Oil, Makeup & Waterproof Mascara, 4 fl. oz',
   'asin': 'B00U2VQZC4',
   'score': 17.792769631882074}],
 'semantic': [{'title': 'GOMAY Hydrating Amino Acid Foaming Facial Cleanser, Daily Face Wash for Makeup Remover with Face Cleansing Brush',
   'asin': 'B09ZDQ626L',
   'score': 0.6298004388809204},
  {'title': 'REBONCEL Aqua Rich Hydrating Face Foam Cleanser Gentle Hypoallergenic pH Balance Korean Skin Care Face Wash Cleansing Foam(8.45 fl.oz 250ml)',
   'asin': 'B09X23VTSQ',
   'score': 0.6638857126235962},
  {'title': 'Hylunia Hydrate Body Wash - Energizing Blend With Mango 8.5 oz',
   'asin': 'B07YNDWRCB',
   'score': 0.7077690958976746},
  {'title': 'Black Wolf - Men’s Gentle Hydrating Face Wash - 5 Fl Oz - Hydrating Sugar Technology Blend Helps Replenish Moisture - Natural Oils Argan, Coconut, & Safflower Hydrate Your Skin, Great for Dry Skin',
   

In [15]:
sample_doc, sample_score = all_results[list(all_results.keys())[0]]["bm25"][0]

print(type(sample_doc))
print(sample_doc)

if hasattr(sample_doc, "model_dump"):
    print(sample_doc.model_dump())
else:
    print(vars(sample_doc))

<class 'langchain_core.documents.base.Document'>
page_content='Neutrogena Ultra Light Facial Cleansing Oil & Makeup Remover, Non-Comedogenic Face Oil Cleanser to Remove Dirt, Oil, Makeup & Waterproof Mascara, 4 fl. oz good facial cleanser my sensitive skin' metadata={'asin': 'B00U2VQZC4', 'product_review': 'Good facial cleanser for my sensitive skin', 'product_title': 'Neutrogena Ultra Light Facial Cleansing Oil & Makeup Remover, Non-Comedogenic Face Oil Cleanser to Remove Dirt, Oil, Makeup & Waterproof Mascara, 4 fl. oz', 'product_rating': 5.0}
{'id': None, 'metadata': {'asin': 'B00U2VQZC4', 'product_review': 'Good facial cleanser for my sensitive skin', 'product_title': 'Neutrogena Ultra Light Facial Cleansing Oil & Makeup Remover, Non-Comedogenic Face Oil Cleanser to Remove Dirt, Oil, Makeup & Waterproof Mascara, 4 fl. oz', 'product_rating': 5.0}, 'page_content': 'Neutrogena Ultra Light Facial Cleansing Oil & Makeup Remover, Non-Comedogenic Face Oil Cleanser to Remove Dirt, Oil, Mak